In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
# import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os
import numpy as np

In [2]:
AUDIO_DIR = "audios/labeled/"

dict_genres = {'positive': 0, 'negative': 1, 'noise': 2}

reverse_map = {v: k for k, v in dict_genres.items()}
print(reverse_map)

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(DEVICE)

{0: 'positive', 1: 'negative', 2: 'noise'}
cuda


In [3]:
data = []

for label in dict_genres.keys():
    for folder in os.listdir(AUDIO_DIR + label):
        for file in os.listdir(AUDIO_DIR + label + "/" + folder):
            file_path = AUDIO_DIR + label + "/" + folder + "/" + file
            positive = int(label == "positive")
            negative = int(label == "negative")
            noise = int(label == "noise")
            data.append((file_path, positive,
                        negative, noise))

file_path, positive, negative, noise = zip(*data)
df = pd.DataFrame({"file_path": file_path, "positive": positive, "negative": negative, "noise": noise})

print(df.head(5))

                                 file_path  positive  negative  noise
0  audios/labeled/positive/1001/100101.wav         1         0      0
1  audios/labeled/positive/1001/100102.wav         1         0      0
2  audios/labeled/positive/1001/100103.wav         1         0      0
3  audios/labeled/positive/1001/100104.wav         1         0      0
4  audios/labeled/positive/1001/100105.wav         1         0      0


In [4]:
class AudioDataset(Dataset):
    def __init__(self, 
                df,
                audio_length = 1, #max length of negative sample = 7.1 seconds
                target_sample_rate=22050):
        self.df = df
        self.file_paths = df['file_path'].values
        self.labels = df[['positive', 'negative']].values
        self.target_sample_rate = target_sample_rate
        self.num_samples = target_sample_rate * audio_length

        self.melspectrogram = T.MelSpectrogram(sample_rate=self.target_sample_rate,
                                               n_mels=128,
                                               n_fft=2048,
                                               hop_length=512)#.to(DEVICE)
        
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):

        # get audio path and label
        audio_path = self.file_paths[index]
        label = self.labels[index]

        # Load audio from file to waveform
        audio, sample_rate = torchaudio.load(audio_path)

        # register to cpu/gpu
        audio = audio#.to(DEVICE)

        # Resample to target sample rate
        audio = self._resample(audio, sample_rate)

        # Convert to mono
        audio = self._mono(audio)

        # Adjust number of samples
        audio = self._crop(audio)
        audio = self._right_pad(audio)

        # Add any preprocessing you like here 
        # (e.g., noise removal, etc.)
        
        
        # Add any data augmentations for waveform you like here
        # (e.g., noise injection, shifting time, changing speed and pitch)
        """ 
        wave_transforms = T.PitchShift(sample_rate, 4)
        audio = wave_transforms(audio)
        """

        # Convert to Mel spectrogram
        melspec = self.melspectrogram(audio)
        
        # Add any data augmentations for spectrogram you like here
        # (e.g., Mixup, cutmix, time masking, frequency masking)
        """ 
        spec_transforms = T.FrequencyMasking(freq_mask_param=80)
        melspec = spec_transforms(melspec)
        """
        return melspec, label

    def _crop(self, audio):
        if audio.shape[1] > self.num_samples:
            audio = audio[:, :self.num_samples]
        return audio
    
    def _right_pad(self, audio):
        audio_length = audio.shape[1]
        if audio_length < self.num_samples:
            num_missing_samples = self.num_samples - audio_length
            last_dim_padding = (0, num_missing_samples)
            audio = F.pad(audio, last_dim_padding)
        return audio

    def _resample(self, audio, sr):
        if sr != self.target_sample_rate:
            resampler = T.Resample(sr, self.target_sample_rate)
            audio = resampler(audio)
        return audio

    def _mono(self, audio):
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)
        return audio

In [5]:
def create_data_loader(train_data, batch_size):
    train_dataloader = DataLoader(train_data, batch_size=batch_size)
    return train_dataloader


def train_single_epoch(model, data_loader, criterion, optimizer):
    model.train()
    train_loss = 0.0
    train_correct = 0
    for input, target in data_loader:
        optimizer.zero_grad()
        input, target = input.to(DEVICE), target.to(DEVICE)

        # calculate loss
        logits = model(input)
        pred_prob = nn.Softmax(dim=1)(logits)
        y_pred = pred_prob.argmax(1)
        loss = criterion(pred_prob, target)

        # backpropagate error and update weights
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"loss: {loss.item()}")


def train(model, data_loader, criterion, optimizer, epochs):
    for i in range(epochs):
        print(f"Epoch {i+1}")
        train_single_epoch(model, data_loader, criterion, optimizer)
        print("---------------------------")
    print("Finished training")

In [6]:
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 0.0005
SAVE = False

dataset = AudioDataset(df)

train_dataloader = create_data_loader(dataset, BATCH_SIZE)

# construct model and assign it to device
model = CRNNNetwork().to(DEVICE)
#summary(model.cuda(), (1, 64, 44))

# initialise loss funtion + optimiser
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters(),
                                lr=LEARNING_RATE)

# train model
train(model, train_dataloader, criterion, optimizer, EPOCHS)

# save model
if SAVE:
    torch.save(model.state_dict(), "feedforwardnet.pth")
    print("Trained feed forward net saved at feedforwardnet.pth")

NameError: name 'CRNNNetwork' is not defined